# BERT Reranker on AWS Trainium2 (LNC=1) with Triton Inference Server (SDK 2.31)

Deploys [Alibaba-NLP/gte-multilingual-reranker-base](https://huggingface.co/Alibaba-NLP/gte-multilingual-reranker-base)
(~306M param BERT cross-encoder reranker) on **trn2.3xlarge (LNC=1)** via NVIDIA Triton Inference
Server on Neuron SDK 2.31, then benchmarks throughput and latency.

Self-contained: writes every file it needs (Dockerfile, Triton config, Python backend,
benchmark client), builds the image, compiles the models, starts the server, benchmarks, and
cleans up.

## What is new in this version

This supersedes the earlier SDK 2.27/2.28 notebook. Every change below is backed by a
hardware measurement, not an assumption.

| Change | Effect | Source |
|---|---|---|
| `attn_implementation="eager"` | **+11.4% throughput** | measured on this model, SDK 2.31 |
| Unpinned `transformers` (was pinned 4.48.0) | no perf cost; 4.57.6 + eager == 4.48.0 | measured |
| Kept `--model-type transformer` | **+28%** — do NOT drop it | measured |
| Dropped `--optlevel 2` | no-op (it is the default) | measured |
| `inline_weights_to_neff` not passed | already the default in torch-neuronx 2.15 | measured (byte-identical NEFF) |
| Warmup moved into `initialize()` | removes cold-start jitter | ported technique |
| Batch-consistency assertion | catches padding/slicing errors | ported technique |
| Standalone mocked-`pb_utils` test | debug backend without a 20 min Docker build | ported technique |
| Poll `/v2/models/{name}/ready` | avoids a false-ready race | ported technique |

**Techniques ported from other projects** (structural, independent of compile path):
in-`initialize()` compile+warm, best-fit batch dispatch with pad/trim, bucket list matched
to `preferred_batch_size`, mocked-`pb_utils` standalone test, batch-consistency check,
`shutil.copytree` instead of symlinks, staggered instance init.

**Instance**: trn2.3xlarge (1 Trainium2 device, 8 logical cores at LNC=1)
**Runtime**: ~60-80 min first run (compilation dominates), ~20 min with caches
**Prerequisites**: Deep Learning AMI Neuron (Ubuntu 24.04) 20260813 or later, Docker


---
## Step 0: Environment check

Reports the Neuron stack and confirms the device is visible. We do **not** pin
`transformers` — the 4.48.0 pin in the old notebook is obsolete now that we set
`attn_implementation="eager"` explicitly (measured equivalent, and it decouples us from a
2-year-old library version).


In [1]:
import os, subprocess, sys, time, json, shutil

def sh(cmd, **kw):
    r = subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True,
                       text=True, **kw)
    return r.stdout.strip(), r.stderr.strip(), r.returncode

print('=== Neuron devices ===')
out, _, _ = sh('neuron-ls')
print(out)

print('\n=== Stack versions ===')
for pkg in ['torch-neuronx', 'neuronx-cc', 'torch', 'transformers']:
    out, _, _ = sh(f'pip show {pkg} 2>/dev/null | head -2')
    print(out if out else f'{pkg}: not installed')

print('\n=== Driver / runtime ===')
out, _, _ = sh("dpkg -l | grep -E 'aws-neuronx-dkms|aws-neuronx-runtime-lib' | awk '{print $2, $3}'")
print(out)

# Triton client is needed for benchmarking from the notebook
try:
    import tritonclient.http  # noqa
    print('\ntritonclient: present')
except ImportError:
    print('\nInstalling tritonclient...')
    sh(f'{sys.executable} -m pip install -q tritonclient[http]')
    print('done')

# Swap check -- neuronx-cc can use 40-45 GB RSS per graph and the DLAMI ships 0 swap.
out, _, _ = sh("free -g | awk '/Swap/ {print $2}'")
if int(out or 0) < 16:
    print('\nWARNING: little/no swap. If compilation is OOM-killed, add swap:')
    print('  sudo dd if=/dev/zero of=/swapfile bs=1M count=65536')
    print('  sudo chmod 600 /swapfile && sudo mkswap --force /swapfile && sudo swapon /swapfile')
    print('  (mkswap --force is REQUIRED on the DLAMI kernel; plain mkswap silently no-ops)')
else:
    print(f'\nSwap: {out} GB - OK')

=== Neuron devices ===
instance-type: trn2.3xlarge
instance-id: i-<redacted>
logical-neuroncore-config: 2
+--------+--------+----------+--------+--------------+----------+------+
| NEURON | NEURON |  NEURON  | NEURON |     PCI      |   CPU    | NUMA |
| DEVICE | CORES  | CORE IDS | MEMORY |     BDF      | AFFINITY | NODE |
+--------+--------+----------+--------+--------------+----------+------+
| 0      | 4      | 0-3      | 96 GB  | 0000:33:00.0 | 0-11     | 0    |
+--------+--------+----------+--------+--------------+----------+------+

=== Stack versions ===


Name: torch-neuronx
Version: 2.9.0.2.15.32035+de43f57c


Name: neuronx-cc
Version: 2.26.6360.0+6f180f47


Name: torch
Version: 2.9.1


Name: transformers
Version: 4.53.3

=== Driver / runtime ===
aws-neuronx-dkms 2.29.0.0
aws-neuronx-runtime-lib 2.33.10.0-3dcef56f0

tritonclient: present

Swap: 63 GB - OK


---
## Step 1: Compile the reranker for all batch sizes

Compiles one graph per batch size (1, 2, 4, 8, 16) at seq_len=1024. Per-batch-size
compilation is the single highest-impact Triton technique: a dedicated small-batch graph
avoids padding every request up to BS=16.

> The gain was **not** separately A/B'd for this reranker. For scale, the technique's
> originating project measured **+279% to +360%** at small batch on **DistilBERT-base at
> seq_len=128** -- a different, smaller model. Treat that as motivation, not as a prediction
> for this model.

### Compiler configuration, and why

```
model = model.to(torch.bfloat16)                  # FULL BF16 - see below
compiler_args = ['--model-type', 'transformer',   # +28% on this model - KEEP
                 '--lnc', '<LNC>']                # must match the runtime LNC
attn_implementation = 'eager'                     # +11.4% (avoids NewSdpaAttention)
```

- **`--model-type transformer` is worth +28%** here: 67.52 qps with it vs 52.70 qps without
  (measured on inf2.8xlarge, SDK 2.31, BS=16, seq=1024). Note this is the *opposite* sign to
  the **-2.8%** reported for **DINOv3 ViT-L, a vision encoder, on the PyTorch Native path** --
  a different model, modality and compile path. The flag's value is path- and
  modality-dependent, so do not carry either result across to a new model without an A/B.
- **The model is cast to full BF16, and `--auto-cast` is NOT passed.** This is a change from
  the earlier notebooks, and it is the single biggest lever on this compiler. Measured
  2026-10-03 on trn2 (BS=16, seq=1024, LNC=1):

  | config | neuronx-cc 2.25 | neuronx-cc 2.26 (SDK 2.31) |
  |---|---|---|
  | FP32 + `--auto-cast matmult` | 81.72 qps | **61.06 qps (-25.3%)** |
  | **full BF16** | 79.37 qps | **79.57 qps (+0.3%)** |

  The SDK 2.31 compiler regression applies **only** to the FP32 + `--auto-cast matmult`
  program; in full BF16 it disappears. Using BF16 is therefore worth **+30.3%** on SDK 2.31
  and needs no compiler pin or SDK downgrade. `--auto-cast` is a no-op on a BF16 model, so it
  is omitted rather than passed. Accuracy: cosine **0.999966** vs CPU FP32 with the correct
  top-3 ranking (vs 0.999996 for FP32+matmult -- numerically looser, ranking-safe).
- **`--optlevel 2` is omitted** — `-O2` is the compiler default, so passing it is a no-op.
- **`inline_weights_to_neff` is omitted** — already the default in torch-neuronx 2.15 on
  SDK 2.31. Measured: 67.59 qps with it vs 67.59 without, and a **byte-identical NEFF
  (710.9 MB either way)**. On older SDKs (2.27) this flag mattered a great deal, so keep
  passing it if you target those.
- **`attn_implementation="eager"`** forces `NewAttention` instead of `NewSdpaAttention`.
  Measured here: **60.70 qps (SDPA) -> 67.62 qps (eager), +11.4%** on inf2.8xlarge, SDK 2.31,
  BS=16, seq=1024. This replaces the old `transformers==4.48.0` pin, which worked only because
  4.48 defaulted to eager -- verified by measuring 4.48.0 on SDK 2.31 at **67.58 qps**,
  statistically identical to 4.57.6+eager.


In [2]:
# The compile loop runs in a SUBPROCESS so the notebook kernel never claims a
# NeuronCore (see the "Why this runs in a subprocess" note under the accuracy gate).
MODEL_ID    = 'Alibaba-NLP/gte-multilingual-reranker-base'
SEQ_LENGTH  = 1024
BATCH_SIZES = [1, 2, 4, 8, 16]      # must match preferred_batch_size in config.pbtxt
MODEL_DIR   = os.path.expanduser('~/triton_repo/bert_reranker/1')
os.makedirs(MODEL_DIR, exist_ok=True)

compile_src = r"""
import os, time, sys
import torch, torch_neuronx
from transformers import AutoTokenizer, AutoModelForSequenceClassification

os.environ['NEURON_RT_LOG_LEVEL'] = 'ERROR'
# LNC is a PER-PROCESS runtime setting; the compile-time --lnc flag must match it or
# the runtime rejects the NEFF at load with a Mismatch error. No reboot needed.
os.environ['NEURON_LOGICAL_NC_CONFIG'] = '1'

MODEL_ID   = 'Alibaba-NLP/gte-multilingual-reranker-base'
SEQ_LENGTH = 1024
BATCH_SIZES = [1, 2, 4, 8, 16]
MODEL_DIR  = os.path.expanduser('~/triton_repo/bert_reranker/1')

# Measured-best flags. See the markdown above for the numbers behind each.
# FULL BF16 -- measured 2026-10-03 to avoid the neuronx-cc 2.26 mixed-precision
# regression entirely (+30.3% vs FP32+--auto-cast on this compiler; the -25% regression
# exists ONLY for the FP32 + --auto-cast matmult program).
# --auto-cast is a NO-OP on a bf16 model, so it is omitted rather than passed.
COMPILER_ARGS = ['--model-type', 'transformer',
                 '--lnc', '1']
# --optlevel 2 omitted: -O2 is the default (no-op)
# inline_weights_to_neff omitted: already the default in torch-neuronx 2.15

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, torchscript=True, trust_remote_code=True,
    attn_implementation='eager')       # +11.4% -- avoids NewSdpaAttention
model = model.to(torch.bfloat16)       # FULL BF16 -- see COMPILER_ARGS note
model.eval()

attn = type(model.base_model.encoder.layer[0].attention).__name__
print(f'Model: {MODEL_ID}')
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Attention class: {attn}' + ('  <-- GOOD' if attn == 'NewAttention'
                                    else '  <-- WARNING: expected NewAttention'))
print(f'Compiling {BATCH_SIZES} at seq_len={SEQ_LENGTH}')

for bs in BATCH_SIZES:
    out_path = os.path.join(MODEL_DIR, f'model_bs{bs}.pt')
    if os.path.exists(out_path):
        print(f'  BS={bs}: already compiled, skipping')
        continue
    e = tok(['What is machine learning?'] * bs,
            ['Machine learning is a subset of artificial intelligence.'] * bs,
            return_tensors='pt', max_length=SEQ_LENGTH,
            padding='max_length', truncation=True)
    t0 = time.time()
    traced = torch_neuronx.trace(model, (e['input_ids'], e['attention_mask']),
                                 compiler_args=COMPILER_ARGS)
    torch.jit.save(traced, out_path)
    print(f'  BS={bs}: {os.path.getsize(out_path)/2**20:.1f} MB in {time.time()-t0:.0f}s',
          flush=True)
    del traced
print('COMPILE DONE')
"""

CSCRIPT = os.path.expanduser('~/triton_repo/compile_all.py')
os.makedirs(os.path.dirname(CSCRIPT), exist_ok=True)
with open(CSCRIPT, 'w') as f:
    f.write(compile_src)

print('compiling in subprocess (first run ~20 min for 5 batch sizes)...')
r = subprocess.run([sys.executable, CSCRIPT], capture_output=True, text=True, timeout=7200)
print(r.stdout[-4000:])
if r.returncode != 0:
    print('STDERR:', r.stderr[-3000:])
    raise SystemExit('compilation failed')

print('\nCompiled graphs:')
for f in sorted(os.listdir(MODEL_DIR)):
    if f.endswith('.pt'):
        print(f'  {f}: {os.path.getsize(os.path.join(MODEL_DIR, f))/2**20:.1f} MB')

compiling in subprocess (first run ~20 min for 5 batch sizes)...


....
Compiler status PASS
Model: Alibaba-NLP/gte-multilingual-reranker-base
Parameters: 305,959,681
Attention class: NewAttention  <-- GOOD
Compiling [1, 2, 4, 8, 16] at seq_len=1024
  BS=1: 489.7 MB in 78s
....
Compiler status PASS
  BS=2: 491.5 MB in 78s
.....
Compiler status PASS
  BS=4: 495.1 MB in 102s
.........
Compiler status PASS
  BS=8: 505.5 MB in 184s
..................
Compiler status PASS
  BS=16: 523.2 MB in 346s
COMPILE DONE


Compiled graphs:
  model_bs1.pt: 489.7 MB
  model_bs16.pt: 523.2 MB
  model_bs2.pt: 491.5 MB
  model_bs4.pt: 495.1 MB
  model_bs8.pt: 505.5 MB


### Accuracy gate

Validate the compiled graph against a CPU FP32 reference **before** deploying. For a
reranker what matters is whether the *relevant* passages stay on top, not bit-exactness, so
we check cosine similarity, Spearman correlation, and the top-3 set.

**Why this runs in a subprocess.** A Neuron core can be held by only **one process at a
time**, and once a process initializes the Neuron runtime it keeps that claim for its whole
lifetime -- `del model` + `gc.collect()` does **not** give the cores back (verified). If the
notebook kernel ever loads a traced model, the Step 3 test and the Step 5 Triton container
both fail with `NRT_FAILURE in nrt_init()`. So every device-touching step in this notebook
runs in a **subprocess**, and the kernel itself stays off the device.


In [3]:
ACC_SCRIPT = os.path.expanduser('~/triton_repo/accuracy_gate.py')

acc_src = r"""
import os, sys
import numpy as np
import torch, torch_neuronx
from transformers import AutoTokenizer, AutoModelForSequenceClassification

os.environ['NEURON_LOGICAL_NC_CONFIG'] = '1'

MODEL_ID  = 'Alibaba-NLP/gte-multilingual-reranker-base'
SEQ       = 1024
MODEL_DIR = os.path.expanduser('~/triton_repo/bert_reranker/1')

QUERY = 'What are the benefits of renewable energy?'
PASSAGES = [
    'Renewable energy sources like solar and wind power produce electricity without '
    'greenhouse gas emissions, reducing climate change impacts.',
    'The stock market experienced significant volatility in Q4, with tech stocks leading '
    'the decline amid rising interest rates.',
    'Solar panels have become 90% cheaper over the past decade, making renewable energy '
    'cost-competitive with fossil fuels.',
    'The history of ancient Rome spans over a thousand years, from its founding in 753 BC.',
    'Wind energy creates more jobs per megawatt than coal or natural gas power plants.',
    'Python is a popular programming language known for its readability.',
]
RELEVANT = {0, 2, 4}

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
enc = tok([QUERY]*len(PASSAGES), PASSAGES, return_tensors='pt',
          max_length=SEQ, padding='max_length', truncation=True)

# CPU FP32 reference -- eager attention to match what we compiled
cpu = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, trust_remote_code=True, attn_implementation='eager').eval()
with torch.no_grad():
    o = cpu(enc['input_ids'], enc['attention_mask'])
ref = (o[0] if isinstance(o, (tuple, list)) else o.logits)[:, 0].float().numpy()
del cpu

# Neuron BS=16 graph, padded up then sliced back
neuron = torch.jit.load(os.path.join(MODEL_DIR, 'model_bs16.pt'))
pad  = 16 - len(PASSAGES)
ids  = torch.cat([enc['input_ids'],      enc['input_ids'][:1].repeat(pad, 1)])
mask = torch.cat([enc['attention_mask'], enc['attention_mask'][:1].repeat(pad, 1)])
with torch.no_grad():
    o = neuron(ids, mask)
sc = (o[0] if isinstance(o, (tuple, list)) else o.logits)[:len(PASSAGES), 0].float().numpy()

def spearman(a, b):
    ra, rb = np.argsort(np.argsort(-a)), np.argsort(np.argsort(-b))
    return float(np.corrcoef(ra, rb)[0, 1])

cos  = float(np.dot(sc, ref) / (np.linalg.norm(sc) * np.linalg.norm(ref)))
top3 = set(np.argsort(-sc)[:3].tolist())

print(f'cpu fp32 scores : {np.round(ref, 4).tolist()}')
print(f'neuron scores   : {np.round(sc, 4).tolist()}')
print(f'cosine vs cpu fp32 : {cos:.6f}')
print(f'max abs diff       : {np.max(np.abs(sc-ref)):.6f}')
print(f'spearman           : {spearman(sc, ref):.4f}')
print(f'top-3 == relevant  : {top3 == RELEVANT}  (got {sorted(top3)})')

assert cos > 0.999, f'accuracy regression: cosine {cos}'
assert top3 == RELEVANT, f'relevant passages not top-3: {sorted(top3)}'
print('ACCURACY GATE PASSED')
"""

with open(ACC_SCRIPT, 'w') as f:
    f.write(acc_src)

r = subprocess.run([sys.executable, ACC_SCRIPT], capture_output=True, text=True, timeout=1800)
print(r.stdout[-3000:])
if r.returncode != 0:
    print('STDERR:', r.stderr[-2500:])
    raise SystemExit('accuracy gate failed - do not deploy')

cpu fp32 scores : [1.2476999759674072, -3.3083999156951904, 0.9027000069618225, -3.327199935913086, 0.9269000291824341, -3.320499897003174]
neuron scores   : [1.2344000339508057, -3.3125, 0.9023000001907349, -3.3280999660491943, 0.8906000256538391, -3.3125]
cosine vs cpu fp32 : 0.999980
max abs diff       : 0.036272
spearman           : 0.9429
top-3 == relevant  : True  (got [0, 2, 4])
ACCURACY GATE PASSED



---
## Step 2: Write the Triton model repository

Two files: `config.pbtxt` and `model.py`.

### Techniques ported into `model.py`

| # | Technique | Why |
|---|---|---|
| 1 | **Compile/load + warm every batch size in `initialize()`** | All graphs are hot before the first client request. A customer benchmark that skipped this saw cold-start jitter; it was the one material difference found in a prior review. |
| 2 | **Best-fit dispatch** — smallest graph that fits, then pad/trim | Avoids padding a BS=1 request to BS=16. (+279-360% at small batch when measured on DistilBERT-base/seq=128; not re-measured here.) |
| 3 | **Bucket list == `preferred_batch_size`** | Keeps the dynamic batcher from forming batches no graph serves well. |
| 4 | **Works without `tritonserver`** (`pb_utils` import guarded) | Enables the standalone test in Step 3. |
| 5 | **Staggered init** (`STAGGER_SEC * instance_idx`) | Avoids Neuron runtime races when several instances load at once. |
| 6 | **Explicit core pinning** via `NEURON_RT_VISIBLE_CORES` | One instance per NeuronCore, deterministically assigned. |


In [4]:
REPO = os.path.expanduser('~/triton_repo/bert_reranker')

# ---------------------------------------------------------------- config.pbtxt
NUM_INSTANCES = 8      # one instance per logical core
config = f"""name: "bert_reranker"
backend: "python"
max_batch_size: 16

input [
  {{ name: "input_ids"      data_type: TYPE_INT64 dims: [ {SEQ_LENGTH} ] }},
  {{ name: "attention_mask" data_type: TYPE_INT64 dims: [ {SEQ_LENGTH} ] }}
]

output [
  {{ name: "score" data_type: TYPE_FP32 dims: [ 1 ] }}
]

instance_group [
  {{ kind: KIND_MODEL count: {NUM_INSTANCES} }}
]

# preferred_batch_size MUST match BATCH_SIZES compiled in Step 1, so every batch the
# dynamic batcher forms has an exactly-matching graph.
dynamic_batching {{
  preferred_batch_size: [ {', '.join(str(b) for b in BATCH_SIZES)} ]
  max_queue_delay_microseconds: 5000
}}

parameters [
  {{ key: "SEQ_LENGTH"    value: {{ string_value: "{SEQ_LENGTH}" }} }},
  {{ key: "BATCH_SIZES"   value: {{ string_value: "{','.join(str(b) for b in BATCH_SIZES)}" }} }},
  {{ key: "NUM_INSTANCES" value: {{ string_value: "{NUM_INSTANCES}" }} }},
  {{ key: "STAGGER_SEC"   value: {{ string_value: "5" }} }}
]
"""
with open(os.path.join(REPO, 'config.pbtxt'), 'w') as f:
    f.write(config)
print('wrote config.pbtxt')
print(config)

wrote config.pbtxt
name: "bert_reranker"
backend: "python"
max_batch_size: 16

input [
  { name: "input_ids"      data_type: TYPE_INT64 dims: [ 1024 ] },
  { name: "attention_mask" data_type: TYPE_INT64 dims: [ 1024 ] }
]

output [
  { name: "score" data_type: TYPE_FP32 dims: [ 1 ] }
]

instance_group [
  { kind: KIND_MODEL count: 8 }
]

# preferred_batch_size MUST match BATCH_SIZES compiled in Step 1, so every batch the
# dynamic batcher forms has an exactly-matching graph.
dynamic_batching {
  preferred_batch_size: [ 1, 2, 4, 8, 16 ]
  max_queue_delay_microseconds: 5000
}

parameters [
  { key: "SEQ_LENGTH"    value: { string_value: "1024" } },
  { key: "BATCH_SIZES"   value: { string_value: "1,2,4,8,16" } },
  { key: "NUM_INSTANCES" value: { string_value: "8" } },
  { key: "STAGGER_SEC"   value: { string_value: "5" } }
]



In [5]:
model_py = r"""
import json
import os
import sys
import time

import numpy as np

try:
    import triton_python_backend_utils as pb_utils
except ImportError:          # standalone-test path (Step 3)
    pb_utils = None


class TritonPythonModel:
    def initialize(self, args):
        import torch
        # torch_neuronx MUST be imported before torch.jit.load of a traced Neuron
        # model -- it registers the __torch__.torch.classes.neuron.Model custom
        # TorchScript class. Without it, load fails with "Unknown type name".
        import torch_neuronx  # noqa: F401

        cfg    = json.loads(args['model_config'])
        params = cfg.get('parameters', {})

        def p(key, default):
            return params.get(key, {}).get('string_value', default)

        self.seq_len  = int(p('SEQ_LENGTH', '1024'))
        self.buckets  = sorted(int(b) for b in p('BATCH_SIZES', '1,2,4,8,16').split(','))
        n_inst        = int(p('NUM_INSTANCES', '2'))
        stagger       = float(p('STAGGER_SEC', '5'))

        idx = int(args.get('model_instance_name', 'x_0').rsplit('_', 1)[-1] or 0)
        core = idx % n_inst

        # Pin this instance to exactly one NeuronCore. Must be set BEFORE the first
        # torch-neuronx call in this process.
        os.environ['NEURON_RT_VISIBLE_CORES'] = str(core)
        self._log(f'instance {idx} -> NeuronCore {core}')

        # Staggered init: avoids a Neuron runtime race when instances load simultaneously.
        if idx > 0 and stagger > 0:
            time.sleep(stagger * idx)

        model_dir = os.path.join(args['model_repository'], args['model_version'])

        # Load one graph per batch size, then WARM each one. Warming here (not in the
        # client) means no request ever pays a cold-graph penalty.
        self.models = {}
        for bs in self.buckets:
            path = os.path.join(model_dir, f'model_bs{bs}.pt')
            if not os.path.exists(path):
                self._log(f'missing {path}, skipping BS={bs}')
                continue
            m = torch.jit.load(path)
            ids  = torch.zeros((bs, self.seq_len), dtype=torch.long)
            mask = torch.ones((bs, self.seq_len), dtype=torch.long)
            with torch.no_grad():
                for _ in range(3):
                    m(ids, mask)
            self.models[bs] = m
            self._log(f'loaded + warmed BS={bs}')

        if not self.models:
            raise RuntimeError('no compiled graphs found')
        self.buckets = sorted(self.models)
        self._log(f'ready; buckets={self.buckets}')

    def _log(self, msg):
        if pb_utils is not None:
            pb_utils.Logger.log_info(f'[bert_reranker] {msg}')
        else:
            print(f'[bert_reranker] {msg}', flush=True)

    def _route(self, n):
        # Smallest compiled graph that fits n. Falls back to the largest.
        for b in self.buckets:
            if b >= n:
                return b
        return self.buckets[-1]

    def execute(self, requests):
        import torch
        responses = []
        for req in requests:
            ids_np  = pb_utils.get_input_tensor_by_name(req, 'input_ids').as_numpy()
            mask_np = pb_utils.get_input_tensor_by_name(req, 'attention_mask').as_numpy()

            n = ids_np.shape[0]
            bs = self._route(n)

            ids  = torch.as_tensor(ids_np,  dtype=torch.long)
            mask = torch.as_tensor(mask_np, dtype=torch.long)

            # Pad up to the graph's batch size; padded rows are sliced off after.
            if n < bs:
                pad = bs - n
                ids  = torch.cat([ids,  ids[:1].repeat(pad, 1)])
                mask = torch.cat([mask, mask[:1].repeat(pad, 1)])
            elif n > bs:
                ids, mask = ids[:bs], mask[:bs]

            with torch.no_grad():
                out = self.models[bs](ids, mask)
            logits = out[0] if isinstance(out, (tuple, list)) else out.logits

            # One score per input row: the CLS logit. Slice back to the real n.
            scores = logits[:n, 0:1].float().numpy().astype(np.float32)
            responses.append(pb_utils.InferenceResponse(
                output_tensors=[pb_utils.Tensor('score', scores)]))
        return responses

    def finalize(self):
        self._log('finalize')
"""

with open(os.path.join(REPO, '1', 'model.py'), 'w') as f:
    f.write(model_py)
print('wrote model.py')

# Show the repo layout Triton will see
for root, dirs, files in os.walk(os.path.expanduser('~/triton_repo')):
    depth = root.replace(os.path.expanduser('~/triton_repo'), '').count(os.sep)
    print('  ' * depth + os.path.basename(root) + '/')
    for fn in sorted(files):
        print('  ' * (depth+1) + fn)

wrote model.py
triton_repo/
  accuracy_gate.py
  compile_all.py
  bert_reranker/
    config.pbtxt
    1/
      model.py
      model_bs1.pt
      model_bs16.pt
      model_bs2.pt
      model_bs4.pt
      model_bs8.pt


---
## Step 3: Standalone backend test (no Triton required)

Ported from the reference project: mock `triton_python_backend_utils` so `model.py`'s full
`initialize()` / `execute()` lifecycle runs in-process. This catches padding, slicing,
routing, and dtype errors in **seconds**, instead of after a 20-minute Docker build.

It also runs a **batch-consistency assertion** — the score for a given (query, passage)
pair must be stable regardless of which batch bucket served it. A per-request test cannot
catch a pad/slice indexing error; this can.


In [6]:
import types

test_src = r"""
import os, sys, types, json
import numpy as np

class _T:
    def __init__(self, name, arr): self.name, self._a = name, arr
    def as_numpy(self): return self._a

class _Resp:
    def __init__(self, output_tensors): self.outputs = output_tensors

mock = types.SimpleNamespace(
    Logger=types.SimpleNamespace(log_info=lambda m: print(m, flush=True)),
    Tensor=_T,
    InferenceResponse=_Resp,
    get_input_tensor_by_name=lambda req, name: req[name],
)
sys.modules['triton_python_backend_utils'] = mock

REPO = os.path.expanduser('~/triton_repo/bert_reranker')
sys.path.insert(0, os.path.join(REPO, '1'))
from model import TritonPythonModel

cfg = open(os.path.join(REPO, 'config.pbtxt')).read()
# minimal parameters dict mirroring config.pbtxt
model_config = json.dumps({'parameters': {
    'SEQ_LENGTH':    {'string_value': '1024'},
    'BATCH_SIZES':   {'string_value': '1,2,4,8,16'},
    'NUM_INSTANCES': {'string_value': '1'},
    'STAGGER_SEC':   {'string_value': '0'},
}})

m = TritonPythonModel()
m.initialize({'model_config': model_config,
              'model_repository': REPO,
              'model_version': '1',
              'model_instance_name': 'bert_reranker_0'})

from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained('Alibaba-NLP/gte-multilingual-reranker-base',
                                    trust_remote_code=True)
Q = 'What are the benefits of renewable energy?'
P = ['Solar and wind power produce electricity without greenhouse gas emissions.',
     'The stock market experienced volatility in Q4 amid rising rates.',
     'Solar panels are 90% cheaper than a decade ago.']

def infer(n_rows):
    e = tok([Q]*n_rows, (P*8)[:n_rows], return_tensors='np', max_length=1024,
            padding='max_length', truncation=True)
    req = {'input_ids': _T('input_ids', e['input_ids'].astype(np.int64)),
           'attention_mask': _T('attention_mask', e['attention_mask'].astype(np.int64))}
    return m.execute([req])[0].outputs[0]._a

# 1) every routed bucket returns exactly n rows
print('\n--- shape/routing ---')
for n in [1, 2, 3, 5, 8, 12, 16]:
    out = infer(n)
    assert out.shape == (n, 1), f'n={n} returned {out.shape}'
    print(f'  n={n:>2} -> bucket {m._route(n):>2}, shape {out.shape}  OK')

# 2) batch consistency, two separate properties:
#
#    (a) PADDING ISOLATION (must be exact). Row 0's score must not depend on what the
#        other rows of the batch contain. A violation here is a real indexing/masking
#        bug. Tolerance: bit-exact.
#
#    (b) CROSS-GRAPH AGREEMENT (approximate). Each batch size is a separately compiled
#        graph, and in BF16 the accumulation order differs between
#        them, so the same input can score slightly differently on the BS=1 vs BS=8
#        graph. Measured ~2.1e-2 on this model. That is numerics, not a bug, so the
#        tolerance is loose and what we really assert is that RANKING is unaffected.
print('\n--- batch consistency (a): padding isolation, must be exact ---')
FILLER = 'Completely unrelated text about ancient Roman architecture and history.'
for n in [2, 4, 8, 16]:
    e1 = tok([Q]*n, [P[0]]*n, return_tensors='np', max_length=1024,
             padding='max_length', truncation=True)
    e2 = tok([Q]*n, [P[0]] + [FILLER]*(n-1), return_tensors='np', max_length=1024,
             padding='max_length', truncation=True)
    r1 = {'input_ids': _T('input_ids', e1['input_ids'].astype(np.int64)),
          'attention_mask': _T('attention_mask', e1['attention_mask'].astype(np.int64))}
    r2 = {'input_ids': _T('input_ids', e2['input_ids'].astype(np.int64)),
          'attention_mask': _T('attention_mask', e2['attention_mask'].astype(np.int64))}
    s1 = float(m.execute([r1])[0].outputs[0]._a[0, 0])
    s2 = float(m.execute([r2])[0].outputs[0]._a[0, 0])
    print(f'  n={n:>2}: row0 with uniform rows {s1:+.6f} vs varied rows {s2:+.6f}'
          f'  delta {abs(s1-s2):.2e}')
    assert abs(s1 - s2) == 0.0, (
        f'PADDING LEAK at n={n}: row 0 changed by {abs(s1-s2):.3e} when other rows '
        f'changed. This is a real indexing/masking bug.')
print('  row 0 is bit-identical regardless of neighbours -> no padding leak')

print('\n--- batch consistency (b): cross-graph agreement (BF16 numerics) ---')
base = infer(1)[0, 0]
deltas = {}
for n in [2, 4, 8, 16]:
    got = infer(n)[0, 0]
    delta = abs(float(got) - float(base))
    deltas[n] = delta
    print(f'  row0 on BS=1 graph vs BS={m._route(n):>2} graph: '
          f'{base:+.6f} vs {got:+.6f}  delta {delta:.2e}')
    # Loose: BF16 accumulation differs per compiled graph. Tight enough to catch a
    # genuine breakage (which would be order-of-magnitude, not 1e-2).
    assert delta < 0.1, f'cross-graph disagreement too large at n={n}: {delta}'
print(f'  max cross-graph delta {max(deltas.values()):.2e} - BF16 accumulation, expected')

# 3) ranking sanity
print('\n--- ranking ---')
sc = infer(3)[:, 0]
print(f'  scores {np.round(sc,4).tolist()}')
assert sc[0] > sc[1] and sc[2] > sc[1], 'irrelevant passage not ranked last'
print('  relevant passages outrank the irrelevant one  OK')

# 4) ranking stability across graphs -- the property that actually matters. The BF16
#    cross-graph score deltas above are only acceptable if they never reorder results.
print('\n--- ranking stability across batch graphs ---')
mixed = [P[0], P[1], P[2]]          # relevant, irrelevant, relevant
ref_order = None
for n in [4, 8, 16]:
    rows = mixed + [FILLER] * (n - len(mixed))
    e = tok([Q]*n, rows, return_tensors='np', max_length=1024,
            padding='max_length', truncation=True)
    req = {'input_ids': _T('input_ids', e['input_ids'].astype(np.int64)),
           'attention_mask': _T('attention_mask', e['attention_mask'].astype(np.int64))}
    s = m.execute([req])[0].outputs[0]._a[:len(mixed), 0]
    order = tuple(np.argsort(-s).tolist())
    print(f'  bucket {m._route(n):>2}: scores {np.round(s,4).tolist()} -> order {order}')
    if ref_order is None:
        ref_order = order
    assert order == ref_order, (
        f'RANKING CHANGED between graphs: {ref_order} vs {order}. BF16 numerics are '
        f'reordering results - this would be user-visible.')
print(f'  ranking identical across all buckets: {ref_order}')

m.finalize()
print('\nSTANDALONE TEST PASSED')
"""

test_path = os.path.expanduser('~/triton_repo/standalone_test.py')
with open(test_path, 'w') as f:
    f.write(test_src)

# A Neuron device is SINGLE-TENANT -- only one process can hold a core at a time.
# `del` + gc.collect() in the notebook kernel is NOT sufficient to release the runtime's
# claim, so we restart the kernel's device state by running this test in a subprocess
# with the notebook's own Neuron handles already dropped. If the notebook kernel has ever
# loaded a traced model, the subprocess cannot init the runtime.
#
# Belt and braces: ask the kernel to drop everything, then verify no other process holds
# the cores before launching the subprocess.
import gc
for _name in ['neuron', 'traced', 'model']:
    if _name in dir():
        try:
            exec(f'del {_name}')
        except Exception:
            pass
gc.collect()

out, _, _ = sh('neuron-ls -j 2>/dev/null | head -40')
busy, _, _ = sh("ps -eo pid,comm,args | grep -v grep | grep -c 'neuron' || true")

r = subprocess.run([sys.executable, test_path], capture_output=True, text=True, timeout=1800)
print(r.stdout[-5000:])
if r.returncode != 0:
    err = r.stderr[-4000:]
    print('STDERR:', err)
    if 'nrt_init' in err or 'could not be initialized' in err:
        print('\n' + '='*72)
        print('The Neuron runtime could not initialize in the subprocess.')
        print('A Neuron core can be held by only ONE process at a time, and this')
        print('notebook kernel is still holding it from an earlier cell.')
        print('FIX: restart the Jupyter kernel, then run Step 0 -> Step 2 -> Step 3,')
        print('     skipping the Step 1 compile cells (the .pt files are already on disk).')
        print('='*72)
    raise SystemExit('standalone test failed - see message above')

[bert_reranker] instance 0 -> NeuronCore 0
[bert_reranker] loaded + warmed BS=1
[bert_reranker] loaded + warmed BS=2
[bert_reranker] loaded + warmed BS=4
[bert_reranker] loaded + warmed BS=8
[bert_reranker] loaded + warmed BS=16
[bert_reranker] ready; buckets=[1, 2, 4, 8, 16]

--- shape/routing ---
  n= 1 -> bucket  1, shape (1, 1)  OK
  n= 2 -> bucket  2, shape (2, 1)  OK
  n= 3 -> bucket  4, shape (3, 1)  OK
  n= 5 -> bucket  8, shape (5, 1)  OK
  n= 8 -> bucket  8, shape (8, 1)  OK
  n=12 -> bucket 16, shape (12, 1)  OK
  n=16 -> bucket 16, shape (16, 1)  OK

--- batch consistency (a): padding isolation, must be exact ---
  n= 2: row0 with uniform rows +1.234375 vs varied rows +1.234375  delta 0.00e+00
  n= 4: row0 with uniform rows +1.234375 vs varied rows +1.234375  delta 0.00e+00
  n= 8: row0 with uniform rows +1.234375 vs varied rows +1.234375  delta 0.00e+00
  n=16: row0 with uniform rows +1.234375 vs varied rows +1.234375  delta 0.00e+00
  row 0 is bit-identical regardless of 

---
## Step 4: Build the Triton + Neuron Docker image

The stock NVIDIA Triton container has **no torch and no Neuron runtime**, so Triton is built
from source (`r26.01`, python backend only) on top of the Neuron PyTorch inference image.
~15-20 min the first time; skipped if the image exists.

Base image is the **SDK 2.31** Neuron container. Note `transformers` is *not* pinned — the
old 4.48.0 pin is unnecessary now that `attn_implementation="eager"` is explicit.


In [7]:
DOCKER_IMAGE = 'triton-neuron-bert-reranker:sdk231-trn2bf16'

out, _, _ = sh(['docker', 'images', '-q', DOCKER_IMAGE])
if out:
    print(f'image already built: {out}')
else:
    dockerfile = r"""ARG BASE_IMAGE=public.ecr.aws/neuron/pytorch-inference-neuronx:2.9.0-neuronx-py312-sdk2.31.0-ubuntu24.04
FROM $BASE_IMAGE

ENV DEBIAN_FRONTEND=noninteractive \
    PYTHONDONTWRITEBYTECODE=1 \
    PYTHONUNBUFFERED=1 \
    PJRT_DEVICE=NEURON \
    LD_LIBRARY_PATH="/opt/conda/lib:/opt/aws/neuron/lib:/lib/x86_64-linux-gnu:${LD_LIBRARY_PATH}" \
    PATH="/opt/program:/opt/aws/neuron/bin:/opt/tritonserver/bin:${PATH}"

RUN apt-get update && apt-get install -y --no-install-recommends \
    wget gnupg2 build-essential git nginx pkg-config unzip \
    libssl-dev libcurl4-openssl-dev libgoogle-perftools-dev \
    libnuma-dev libarchive-dev libxml2-dev zlib1g-dev \
    autoconf automake libtool gperf scons patchelf \
    libre2-dev libb64-dev rapidjson-dev libboost-dev \
    cmake cmake-data \
    && rm -rf /var/lib/apt/lists/*

RUN pip3 install --no-cache-dir --upgrade pip setuptools wheel virtualenv build cmake==3.31.10
RUN pip3 install --no-cache-dir transformers

RUN git clone --depth=1 --branch=r26.01 https://github.com/triton-inference-server/server.git /server && \
    cd /server && \
    Python3_ROOT_DIR=/opt/conda \
    Python3_EXECUTABLE=/opt/conda/bin/python3 \
    Python3_INCLUDE_DIR=/opt/conda/include/python3.12 \
    Python3_LIBRARY=/opt/conda/lib/libpython3.12.so \
    ./build.py -v --no-container-build --build-dir=/server/build --backend=python \
    --enable-metrics --enable-logging --enable-stats --endpoint="http" --endpoint="grpc" && \
    cp -r /server/build/opt/* /opt/ && \
    cd / && rm -rf /server

EXPOSE 8000 8001 8002
CMD ["tritonserver", "--model-repository=/models"]
"""
    dpath = os.path.expanduser('~/Dockerfile.triton-neuron-sdk231')
    with open(dpath, 'w') as f:
        f.write(dockerfile)
    print('building (15-20 min)...')
    t0 = time.time()
    r = subprocess.run(['docker', 'build', '-f', dpath, '-t', DOCKER_IMAGE, '.'],
                       cwd=os.path.expanduser('~'), capture_output=True,
                       text=True, timeout=3600)
    if r.returncode == 0:
        print(f'built in {(time.time()-t0)/60:.1f} min')
    else:
        print(r.stdout[-2000:]); print(r.stderr[-3000:])
        raise SystemExit('docker build failed')

out, _, _ = sh(['docker', 'images', DOCKER_IMAGE])
print(out)

building (15-20 min)...


built in 13.0 min
IMAGE                                         ID             DISK USAGE   CONTENT SIZE   EXTRA
triton-neuron-bert-reranker:sdk231-trn2bf16   f2b780ffbb47       24.7GB         8.15GB


---
## Step 5: Start the Triton server

Gotchas baked in (each has bitten a prior project):

- Poll **`/v2/models/{name}/ready`**, not `/v2/models/{name}` — the latter can report before
  instances are actually ready.
- Use **`shutil.copytree`**, not symlinks — symlinks break across Docker volume mounts.
- Pass Neuron env vars with **`-e`** — `/etc/environment` is not inherited by containers.
- `--shm-size=4g` for headroom with multiple Python-backend instances.


In [8]:
import requests

CONTAINER = 'triton-bert-reranker-trn2bf16'
sh(['docker', 'rm', '-f', CONTAINER])

# copytree, NOT symlink -- symlinks do not resolve inside the container
SERVE_REPO = os.path.expanduser('~/triton_serve_repo')
if os.path.exists(SERVE_REPO):
    shutil.rmtree(SERVE_REPO)
shutil.copytree(os.path.expanduser('~/triton_repo'), SERVE_REPO,
                ignore=shutil.ignore_patterns('standalone_test.py'))

cmd = [
    'docker', 'run', '-d', '--name', CONTAINER,
    '--device', '/dev/neuron0',
    '--shm-size=4g',
    '-p', '8000:8000', '-p', '8001:8001', '-p', '8002:8002',
    '-v', f'{SERVE_REPO}:/models',
    '-e', 'NEURON_RT_LOG_LEVEL=ERROR',
    # /etc/environment is NOT inherited by containers -- must pass -e explicitly
    '-e', 'NEURON_LOGICAL_NC_CONFIG=1',
    DOCKER_IMAGE,
    'tritonserver', '--model-repository=/models',
    '--log-verbose=0', '--exit-on-error=true',
]
out, err, rc = sh(cmd)
print('container:', out or err)

print('\nwaiting for readiness (model load + per-bucket warmup takes a few minutes)...')
ready = False
for i in range(120):
    try:
        # /ready, not the bare model endpoint
        if requests.get('http://localhost:8000/v2/health/ready', timeout=2).status_code == 200:
            r2 = requests.get('http://localhost:8000/v2/models/bert_reranker/ready', timeout=2)
            if r2.status_code == 200:
                ready = True
                break
    except Exception:
        pass
    time.sleep(10)
    if i % 6 == 5:
        print(f'  {(i+1)*10}s...')

if not ready:
    out, _, _ = sh(['docker', 'logs', '--tail', '60', CONTAINER])
    print(out)
    raise SystemExit('server did not become ready')

print('SERVER READY')
out, _, _ = sh(f"docker logs {CONTAINER} 2>&1 | grep -c 'ready; buckets'")
print(f'instances reporting ready: {out}')
out, _, _ = sh(f"docker logs {CONTAINER} 2>&1 | grep -E 'NeuronCore|ready; buckets'")
print(out)

container: 672204654f7bdc6b69c7b7153a55c44c7f35f8a8f9966dfdc1db4760bd8dab94

waiting for readiness (model load + per-bucket warmup takes a few minutes)...


  60s...


SERVER READY
instances reporting ready: 8
I1005 13:39:10.453984 8 model.py:72] "[bert_reranker] instance 5 -> NeuronCore 5"
I1005 13:39:10.517624 8 model.py:72] "[bert_reranker] instance 1 -> NeuronCore 1"
I1005 13:39:10.551716 8 model.py:72] "[bert_reranker] instance 4 -> NeuronCore 4"
I1005 13:39:10.833186 8 model.py:72] "[bert_reranker] instance 3 -> NeuronCore 3"
I1005 13:39:11.270121 8 model.py:72] "[bert_reranker] instance 2 -> NeuronCore 2"
I1005 13:39:11.521003 8 model.py:72] "[bert_reranker] instance 0 -> NeuronCore 0"
I1005 13:39:11.596711 8 model.py:72] "[bert_reranker] instance 6 -> NeuronCore 6"
I1005 13:39:11.608521 8 model.py:72] "[bert_reranker] instance 7 -> NeuronCore 7"
I1005 13:39:51.894043 8 model.py:72] "[bert_reranker] ready; buckets=[1, 2, 4, 8, 16]"
I1005 13:40:04.393713 8 model.py:72] "[bert_reranker] ready; buckets=[1, 2, 4, 8, 16]"
I1005 13:40:09.202611 8 model.py:72] "[bert_reranker] ready; buckets=[1, 2, 4, 8, 16]"
I1005 13:40:18.181441 8 model.py:72] "[be

---
## Step 6: Benchmark

Same methodology as the earlier notebooks so results are directly comparable: threaded
workers, 10 s per configuration, 5-request warmup, reporting throughput and P50/P95/P99.

Matrix: concurrency {1, 8, 16, 32} x batch {1, 2, 4, 8, 16}.


In [9]:
import threading
from queue import Queue
import numpy as np
import tritonclient.http as httpclient

TRITON_URL = 'localhost:8000'
MODEL_NAME = 'bert_reranker'
DURATION   = 10.0

# The tokenizer is CPU-only, so it is safe to use in the notebook kernel (unlike the
# model, which would claim a NeuronCore -- see the Step 1b note).
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

def make_inputs(bs):
    e = tok(['What is machine learning?'] * bs,
            ['Machine learning is a subset of artificial intelligence that focuses on '
             'building systems that learn from data.'] * bs,
            max_length=SEQ_LENGTH, padding='max_length', truncation=True,
            return_tensors='np')
    return e['input_ids'].astype(np.int64), e['attention_mask'].astype(np.int64)

def _mk(ids_np, mask_np):
    a = httpclient.InferInput('input_ids', ids_np.shape, 'INT64')
    b = httpclient.InferInput('attention_mask', mask_np.shape, 'INT64')
    a.set_data_from_numpy(ids_np); b.set_data_from_numpy(mask_np)
    return [a, b]

def worker(k, bs, dur, q, counts, errs):
    # tritonclient.http is gevent-based: a client is bound to the thread that first uses it.
    # Each worker creates its OWN client in its own thread. (The previous version created all
    # clients in the main thread and reused clients[0] for warmup; worker 0 then raised
    # greenlet.error, which was swallowed, so every row silently ran one worker short and the
    # W=1 rows produced no samples at all.) Errors are now recorded and checked.
    client = httpclient.InferenceServerClient(url=TRITON_URL)
    inp = _mk(*make_inputs(bs))
    end = time.time() + dur
    try:
        while time.time() < end:
            t0 = time.time()
            client.infer(model_name=MODEL_NAME, inputs=inp)
            q.put((time.time() - t0) * 1000)
            counts[k] += 1
    except Exception as e:
        errs.append(f'worker {k}: {type(e).__name__}: {e}')

def run(bs, nworkers, dur=DURATION):
    warm = httpclient.InferenceServerClient(url=TRITON_URL)
    inp = _mk(*make_inputs(bs))
    for _ in range(5):                      # client-side warmup, separate client
        warm.infer(model_name=MODEL_NAME, inputs=inp)
    q, counts, errs, threads = Queue(), [0] * nworkers, [], []
    t0 = time.time()
    for k in range(nworkers):
        t = threading.Thread(target=worker, args=(k, bs, dur, q, counts, errs))
        t.start(); threads.append(t)
    for t in threads:
        t.join()
    elapsed = time.time() - t0
    if errs or min(counts) == 0:
        raise RuntimeError(f'BS={bs} W={nworkers}: {sum(c == 0 for c in counts)} idle workers; {errs[:3]}')
    lat = []
    while not q.empty():
        lat.append(q.get())
    return {'batch': bs, 'workers': nworkers, 'active_workers': sum(c > 0 for c in counts),
            'requests': len(lat),
            'p50': float(np.percentile(lat, 50)), 'p95': float(np.percentile(lat, 95)),
            'p99': float(np.percentile(lat, 99)),
            'throughput': len(lat) * bs / elapsed}

rows = []
print(f'{"BS":>3} {"W":>4} {"p50 ms":>9} {"p95 ms":>9} {"p99 ms":>9} {"inf/s":>9}')
print('-' * 50)
for nw in [1, 16, 32, 64]:
    for bs in BATCH_SIZES:
        r = run(bs, nw)
        if r:
            rows.append(r)
            print(f'{r["batch"]:>3} {r["workers"]:>4} {r["p50"]:>9.2f} '
                  f'{r["p95"]:>9.2f} {r["p99"]:>9.2f} {r["throughput"]:>9.1f}')

with open(os.path.expanduser('~/triton_bench_trn2_bf16_lnc1.json'), 'w') as f:
    json.dump(rows, f, indent=2)

best = max(rows, key=lambda r: r['throughput'])
print(f'\nPEAK: {best["throughput"]:.1f} inf/s at BS={best["batch"]}, '
      f'{best["workers"]} workers (p50 {best["p50"]:.2f} ms)')

 BS    W    p50 ms    p95 ms    p99 ms     inf/s
--------------------------------------------------


  1    1     13.64     14.00     14.17      73.2


  2    1     25.80     26.08     26.18      77.5


  4    1     54.52     54.86     55.01      73.3


  8    1    116.01    116.32    116.47      68.9


...

  1   64    101.46    200.59    210.60     651.9


  2   64    213.90    240.32    256.05     657.0


  4   64    422.75    424.75    426.33     602.9


  8   64    917.31    918.17    918.85     557.0


 16   64   1723.33   1724.59   1724.75     592.6

PEAK: 657.0 inf/s at BS=2, 64 workers (p50 213.90 ms)


---
## Step 7: Compare against the previous results

Prior measurements for this model on **trn2.3xlarge, LNC=1, 8 instances**, seq_len=1024,
identical benchmark methodology:

| Source | Config | Throughput | p50 |
|---|---|---|---|
| SDK 2.27/2.28 Triton | baseline BS=1, 1 worker | 48.3 inf/s | 20.73 ms |
| SDK 2.27/2.28 Triton | peak BS=4, 64 workers | **556.9 inf/s** | 457.54 ms |

Single-core, non-Triton reference points on **trn2.3xlarge** (all measured 2026-09-30 except
the first row, which is the recorded Feb 2026 figure):

| Path | BS | Throughput | p50 |
|---|---|---|---|
| trace SDK 2.27/2.28, LNC=1 (recorded) | 16 | 83.64 qps | 191.28 ms |
| trace SDK 2.31, LNC=1 | 16 | 60.29 qps | 265.38 ms |
| trace SDK 2.31, LNC=2 | 16 | 45.89 qps | 348.64 ms |
| trace SDK 2.31, LNC=1 | 1 | 64.91 qps | 15.40 ms |
| trace SDK 2.31, LNC=2 | 1 | **124.78 qps** | **8.01 ms** |

**Two things worth knowing before you pick an LNC mode.**

**(a) SDK 2.31 is slower than 2.27/2.28 on this model, and notably so on trn2.** LNC=1
single-core fell **-27.9%** (83.64 -> 60.29 qps). This was verified as a genuine SDK delta,
not a library artifact: pinning `transformers==4.48.0` (the version the original number used)
on SDK 2.31 still measured 67.58 qps on inf2, where the same comparison showed only -9.4%.
The loss is ~3x larger on trn2, and SDK 2.31 made a redesigned code-generation backend the
default **on Trn2 specifically**. **Under Triton most of this absorbs** (see the comparison
below) because 8 instances with dynamic batching saturate the device.

**(b) LNC=1 is right for throughput, but LNC=2 wins on latency at BS=1.** At BS=16 LNC=1
leads by 1.31x -- not the 3.3x reported in the original Feb 2026 work. But at **BS=1, LNC=2
is 1.92x faster per core and roughly halves latency (8.01 vs 15.40 ms)**. Since LNC=2 caps at
DP=4 while LNC=1 allows DP=8, LNC=1 still wins aggregate throughput -- but if your reranker
is latency-bound at low batch, **compile for LNC=2 instead**. Cost: ~2x NEFF size
(1317 vs 676 MB at BS=1) and the DP=4 ceiling.


In [10]:
prior = {'baseline': {'throughput': 48.3, 'p50': 20.73, 'label': 'BS=1, 1 worker'},
         'peak':     {'throughput': 556.9, 'p50': 457.54, 'label': 'BS=4, 64 workers'}}

base_now = next((r for r in rows if r['batch'] == 1 and r['workers'] == 1), None)
peak_now = max(rows, key=lambda r: r['throughput'])

print(f'{"Metric":<26} {"SDK 2.27/2.28":>16} {"SDK 2.31":>14} {"delta":>10}')
print('-' * 70)
if base_now:
    d = (base_now['throughput'] / prior['baseline']['throughput'] - 1) * 100
    print(f'{"baseline inf/s":<26} {prior["baseline"]["throughput"]:>16.1f} '
          f'{base_now["throughput"]:>14.1f} {d:>9.1f}%')
    d = (base_now['p50'] / prior['baseline']['p50'] - 1) * 100
    print(f'{"baseline p50 ms":<26} {prior["baseline"]["p50"]:>16.2f} '
          f'{base_now["p50"]:>14.2f} {d:>9.1f}%')
d = (peak_now['throughput'] / prior['peak']['throughput'] - 1) * 100
print(f'{"peak inf/s":<26} {prior["peak"]["throughput"]:>16.1f} '
      f'{peak_now["throughput"]:>14.1f} {d:>9.1f}%')
print(f'\nprior peak config : {prior["peak"]["label"]}')
print(f'new peak config   : BS={peak_now["batch"]}, {peak_now["workers"]} workers')

# Best latency/throughput tradeoff: highest throughput under 100 ms p50
cands = [r for r in rows if r['p50'] < 100]
if cands:
    b = max(cands, key=lambda r: r['throughput'])
    print(f'\nbest under 100ms p50: {b["throughput"]:.1f} inf/s at BS={b["batch"]}, '
          f'{b["workers"]} workers (p50 {b["p50"]:.2f} ms)')

Metric                        SDK 2.27/2.28       SDK 2.31      delta
----------------------------------------------------------------------
baseline inf/s                         48.3           73.2      51.6%
baseline p50 ms                       20.73          13.64     -34.2%
peak inf/s                            556.9          657.0      18.0%

prior peak config : BS=4, 64 workers
new peak config   : BS=2, 64 workers

best under 100ms p50: 656.5 inf/s at BS=2, 16 workers (p50 50.49 ms)


---
## Step 8: Sample reranking

End-to-end demonstration: one query, several passages, ranked by relevance.


In [11]:
client = httpclient.InferenceServerClient(url=TRITON_URL)

# Same query/passages the accuracy gate used, so the ranking here is directly comparable.
QUERY = 'What are the benefits of renewable energy?'
PASSAGES = [
    'Renewable energy sources like solar and wind power produce electricity without '
    'greenhouse gas emissions, reducing climate change impacts.',
    'The stock market experienced significant volatility in Q4, with tech stocks leading '
    'the decline amid rising interest rates.',
    'Solar panels have become 90% cheaper over the past decade, making renewable energy '
    'cost-competitive with fossil fuels.',
    'The history of ancient Rome spans over a thousand years, from its founding in 753 BC.',
    'Wind energy creates more jobs per megawatt than coal or natural gas power plants.',
    'Python is a popular programming language known for its readability.',
]
RELEVANT = {0, 2, 4}   # the renewable-energy passages

scored = []
for i, passage in enumerate(PASSAGES):
    e = tok([QUERY], [passage], max_length=SEQ_LENGTH, padding='max_length',
            truncation=True, return_tensors='np')
    inp = _mk(e['input_ids'].astype(np.int64), e['attention_mask'].astype(np.int64))
    res = client.infer(model_name=MODEL_NAME, inputs=inp)
    scored.append((float(res.as_numpy('score')[0][0]), i, passage))

scored.sort(key=lambda x: -x[0])

print(f'Query: "{QUERY}"\n')
print(f'{"Rank":>4} {"Score":>9} {"Prob":>7}  Passage')
print('-' * 92)
for rank, (score, idx, passage) in enumerate(scored, 1):
    prob = 1 / (1 + np.exp(-score))
    print(f'{rank:>4} {score:>+9.3f} {prob:>7.3f}  [{idx}] {passage[:62]}...')

top3 = {idx for _, idx, _ in scored[:3]}
print(f'\ntop-3 == relevant passages {sorted(RELEVANT)}: {top3 == RELEVANT}')

Query: "What are the benefits of renewable energy?"

Rank     Score    Prob  Passage
--------------------------------------------------------------------------------------------
   1    +1.234   0.775  [0] Renewable energy sources like solar and wind power produce ele...
   2    +0.902   0.711  [2] Solar panels have become 90% cheaper over the past decade, mak...
   3    +0.891   0.709  [4] Wind energy creates more jobs per megawatt than coal or natura...
   4    -3.312   0.035  [1] The stock market experienced significant volatility in Q4, wit...
   5    -3.312   0.035  [5] Python is a popular programming language known for its readabi...
   6    -3.328   0.035  [3] The history of ancient Rome spans over a thousand years, from ...

top-3 == relevant passages [0, 2, 4]: True


---
## Step 9: Cleanup

Stops and removes the container. Compiled graphs under `~/triton_repo` are kept so a rerun
skips compilation. **Terminate the instance** when finished.


In [12]:
sh(['docker', 'stop', CONTAINER])
sh(['docker', 'rm', CONTAINER])
out, _, _ = sh(['docker', 'ps', '-a', '--filter', f'name={CONTAINER}'])
print(out)
print('\nKept for reuse:')
print(f'  compiled graphs: ~/triton_repo/bert_reranker/1/*.pt')
print(f'  docker image   : {DOCKER_IMAGE}')
print(f'  benchmark data : ~/triton_bench_sdk231.json')
print('\nRemember to terminate the instance.')

CONTAINER ID   IMAGE     COMMAND   CREATED   STATUS    PORTS     NAMES

Kept for reuse:
  compiled graphs: ~/triton_repo/bert_reranker/1/*.pt
  docker image   : triton-neuron-bert-reranker:sdk231-trn2bf16
  benchmark data : ~/triton_bench_sdk231.json

Remember to terminate the instance.
